# Depth revision code


In [ ]:
import flexiznam as flz
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import json
import random
import os
from tqdm import tqdm
import pickle
from scipy.stats import gaussian_kde
import cottage_analysis.analysis.is_learned as learn



In [ ]:
roicat_sessions = {
    'PZAG16.3c': ['S20250219', 'S20250313'], 
    'PZAG17.3a': ['S20250227', 'S20250228', 'S20250303', 'S20250305', 'S20250306'], 
    'PZAG16.3b': ['S20250224', 'S20250225', 'S20250226', 'S20250310', 'S20250313'], 
    'PZAH17.1e': ['S20250305', 'S20250311']
}
learn.run_all_pairs(roicat_sessions)


### Is depth selectivity innate?

In [ ]:
PROJECT='colasa_3d-vision_revisions'
SESSION='PZAG16.3c_S20250219'

In [ ]:
flz_session = flz.get_flexilims_session(PROJECT)

In [ ]:
suite2p_datasets = flz.get_datasets(
        origin_name=SESSION,
        dataset_type="suite2p_rois",
        project_id=PROJECT,
        flexilims_session=flz_session,
        return_dataseries=False,
    )

In [ ]:
dataset = suite2p_datasets[0]

In [ ]:
path = Path('/nemo/lab/znamenskiyp/home/shared/projects/colasa_3d-vision_revisions/PZAG16.3c/S20250219')

In [ ]:
datapath = path / 'neurons_df.pickle'

In [ ]:
neurons_df = pd.read_pickle(datapath)

In [ ]:
neurons_df.columns

In [ ]:
plt.hist(neurons_df['depth_neuron_anova_p'], bins = 100)
plt.axvline(0.05, color = 'red')

In [ ]:
plt.hist(neurons_df['preferred_depth_closedloop'][neurons_df['is_depth_neuron']==True])

In [ ]:
plt.hist(neurons_df['preferred_depth_closedloop'][neurons_df['depth_neuron_anova_p']<0.00001])

In [ ]:
sig_neurons_df = neurons_df[neurons_df['is_depth_neuron']].copy()
sig_neurons_df['log_pref_depth'] = np.log10(neurons_df['preferred_depth_closedloop']*100)
plt.hist(sig_neurons_df['log_pref_depth'], bins = 100)
plt.ylabel('Frequency')
plt.xlabel('log preferred depth (cm)')
plt.title('PZAG16.3c, session 0, significantly depth tuned neurons')

In [ ]:
plt.scatter(neurons_df['preferred_depth_closedloop'], neurons_df['depth_neuron_anova_p'], alpha = 0.5)

In [ ]:

# Count occurrences of True and False in the 'is_depth_neuron' column
value_counts = neurons_df['is_depth_neuron'].value_counts()

# Create a bar plot
plt.figure(figsize=(6, 4))
plt.bar(value_counts.index.astype(str), value_counts.values)

In [ ]:
plt.hist(sig_neurons_df['depth_tuning_test_spearmanr_rval_closedloop'])

In [ ]:
plt.hist(neurons_df['depth_tuning_test_spearmanr_rval_closedloop'])

## Trying the baseline dist

In [ ]:
baseline_datapath = Path('/nemo/lab/znamenskiyp/home/shared/projects/hey2_3d-vision_foodres_20220101/PZAH8.2i/S20230209')

In [ ]:
bas_datapath = baseline_datapath / 'neurons_df.pickle'

In [ ]:
baseline_df = pd.read_pickle(bas_datapath)

In [ ]:
baseline_df.columns

In [ ]:
plt.hist(baseline_df['preferred_depth_closedloop'])

In [ ]:
plt.hist(baseline_df['preferred_depth_closedloop'][baseline_df['is_depth_neuron']==True])

In [ ]:
bas_sig_neurons_df = baseline_df[baseline_df['is_depth_neuron']].copy()
bas_sig_neurons_df['log_pref_depth'] = np.log10(bas_sig_neurons_df['preferred_depth_closedloop']*100)
plt.hist(bas_sig_neurons_df['log_pref_depth'], bins = 100)
plt.ylabel('Frequency')
plt.xlabel('log preferred depth (cm)')
plt.title('PZAH8.2i, session N, significantly depth tuned neurons')

# Finding back the ROIs

In [ ]:
datapath = dataset.path_full / 'plane0'

In [ ]:
iscell = np.load(datapath / 'iscell.npy')

In [ ]:
iscell.shape

# Selectivity across days (no ROIs)

- We want to plot the histograms of depth selectivities over days, to check if there are any changes.
- That is, however we plot it, a dataframe that has sessions in columns, nday, mouse, and a set of bins that have a set proportion of cells.
- Access all the dataframes of mice, then bin the depths and keep those numbers



In [ ]:
PROJECT='colasa_3d-vision_revisions'
flz_session = flz.get_flexilims_session(PROJECT)


In [ ]:
micelist = ['PZAG16.3b', 'PZAG16.3c', 'PZAH17.1e']

### Thinking about how to build the code

In [ ]:
sessions = flz.get_children(
    parent_name = micelist[0], 
    children_datatype = 'session',
    project_id = PROJECT, 
    flexilims_session= flz_session
)
        

In [ ]:
processed_root = flz.get_data_root('processed', 
                                   project=PROJECT, 
                                   flexilims_session=flz_session
                                  )


In [ ]:
sesspath = processed_root / sessions.path[0]

In [ ]:
SphereTube_recordings = flz.get_children(
    parent_name = sessions.name[0], 
    children_datatype = 'recording', 
    project_id=PROJECT, 
    flexilims_session=flz_session
)


In [ ]:
SphereTube_recordings = SphereTube_recordings[SphereTube_recordings['protocol']=='SpheresPermTubeReward']

In [ ]:
recordings = []
for i in sessions.name:
    suite2p_datasets = flz.get_datasets(
        origin_name=i,
        dataset_type="suite2p_rois",
        project_id=PROJECT,
        flexilims_session=flz_session,
        return_dataseries=False,
    )
    print(suite2p_datasets)
    if suite2p_datasets != []:
        recordings.append(suite2p_datasets)
    

In [ ]:
recordings[0][0].path.parent

In [ ]:
def find_processed_sessions(mouse, protocol = 'SpheresPermTubeReward'):
    #Check all sessions
    sessions = flz.get_children(
        parent_name = mouse, 
        children_datatype = 'session',
        project_id = PROJECT, 
        flexilims_session= flz_session
    )
    #print(sessions.name)
    #List the sessions that are SphereTube
    for i in sessions.name:
        SphereTube_recordings = flz.get_children(
            parent_name = i, 
            children_datatype = 'recording', 
            project_id=PROJECT, 
            flexilims_session=flz_session
        )
        SphereTube_recordings = SphereTube_recordings[SphereTube_recordings['protocol']=='SpheresPermTubeReward']
        if len(SphereTube_recordings)==0:
            sessions = sessions[sessions['name']!= i]

    #Keep the sessions that are processed
    for i, session in sessions.iterrows():
        #print(session)
        neurons_path = processed_root / session.path / 'neurons_df.pickle'
        if not os.path.isfile(neurons_path):
            name_to_drop = session.name
            sessions = sessions[sessions['name']!= name_to_drop]
            
    print(sessions.name)
    return sessions

In [ ]:
mouse_i = 0
for mouse in micelist:
    if mouse_i==0:
        processed = find_processed_sessions(mouse)
    else:
        processed = pd.concat([find_processed_sessions(mouse), processed], ignore_index=True)
    mouse_i += 1
    
processed

In [ ]:
def save_significant_neurons(neurons_df):
    '''
    
    '''
    # Filter significantly depth-tuned neurons
    sig_neurons_df = neurons_df[neurons_df['is_depth_neuron']].copy()

    # Compute log preferred depth (in cm)
    sig_neurons_df['log_pref_depth'] = np.log10(sig_neurons_df['preferred_depth_closedloop'] * 100)

    return sig_neurons_df

def save_histogram(sig_neurons_df, session, path, print_figure = True):
    """
    Saves a histogram of the log preferred depth of significantly depth-tuned neurons 
    and returns an alternative histogram with 10 bins as a NumPy array.
    
    Args:
        neurons_df (pd.DataFrame): DataFrame containing neuron data.
        mouse (str): Mouse identifier.
        session (int): Session number.
        path (str, optional): Directory to save the figure. Defaults to "figures/".
    
    Returns:
        np.array: Histogram data with 10 bins.
    """

    # Ensure path exists
    save_path = Path(path)
    save_path.mkdir(parents=True, exist_ok=True)

    if print_figure:
        # Save the histogram with 100 bins
        plt.figure(figsize=(8, 6))
        plt.hist(sig_neurons_df['log_pref_depth'], bins=100, color="blue", alpha=0.7)
        plt.ylabel('Frequency')
        plt.xlabel('Log Preferred Depth (cm)')
        plt.title(f'Session {session}, Significantly Depth-Tuned Neurons')
        
        # Save figure
        filename = save_path / f"session_{session}_hist.png"
        plt.savefig(filename, dpi=300)
        plt.close()

    # Generate alternative histogram with 10 bins
    hist_counts, bin_edges = np.histogram(sig_neurons_df['log_pref_depth'], bins=10)

    return hist_counts, bin_edges

class NumpyFixUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == 'numpy._core.numeric':
            module = 'numpy.core.numeric'
        return super().find_class(module, name)


In [ ]:
hist_counts = []
bin_edges = []
mean_log_depths = []
sems = []

for i, session in tqdm(processed.iterrows()):
    neurons_path = processed_root / session.path / 'neurons_df.pickle'
    save_path = processed_root / session.path / 'figures/'

    with open(neurons_path, 'rb') as f:
        neurons_df = NumpyFixUnpickler(f).load()

    sig_neurons_df = save_significant_neurons(neurons_df)

    # Compute mean and sem of log_pref_depth for significant neurons
    mean_log_depth = np.mean(sig_neurons_df['log_pref_depth'])
    sem = np.std(sig_neurons_df['log_pref_depth'], ddof=1) / np.sqrt(len(sig_neurons_df))

    # Save values
    mean_log_depths.append(mean_log_depth)
    sems.append(sem)

    # Get histogram
    hist_count, bin_edge = save_histogram(
        sig_neurons_df, session.name, save_path, print_figure=False
    )
    hist_counts.append(hist_count)
    bin_edges.append(bin_edge)

# Assign all data to the dataframe
processed['hist_counts'] = hist_counts
processed['bin_edges'] = bin_edges
processed['mean_log_depth'] = mean_log_depths
processed['sem_log_depth'] = sems




In [ ]:
name = processed.name[0]
namelist = name.split('_')
mouse = namelist[0]
date = namelist[1]

In [ ]:
mice = []
dates = []
for i, session in processed.iterrows():
    name = session['name']
    namelist = name.split('_')
    mice.append(namelist[0])
    dates.append(namelist[1])

processed['mouse'] = mice
processed['date'] =  dates

processed_mice = list(set(processed['mouse']))


In [ ]:

# Ensure 'date' is treated as a string or int for sorting (yyyymmdd format is naturally sortable)
processed = processed.sort_values(by=['mouse', 'date']).reset_index(drop=True)

# Create the exposure_day column by grouping by mouse and ranking the date
processed['exposure_day'] = (
    processed
    .groupby('mouse')['date']
    .rank(method='dense')  # or method='first' if dates could repeat
    .astype(int) - 1        # Start from 0
)

In [ ]:
def build_sessions_df(micelist):
    
    #generate the barebones processed dataframe
    mouse_i = 0
    for mouse in micelist:
        if mouse_i==0:
            processed = find_processed_sessions(mouse)
        else:
            processed = pd.concat([find_processed_sessions(mouse), processed], ignore_index=True)
        mouse_i += 1
    
    #add variable values
    hist_counts = []
    bin_edges = []
    mean_log_depths = []
    sems = []

    for i, session in tqdm(processed.iterrows()):
        neurons_path = processed_root / session.path / 'neurons_df.pickle'
        save_path = processed_root / session.path / 'figures/'

        with open(neurons_path, 'rb') as f:
            neurons_df = NumpyFixUnpickler(f).load()

        sig_neurons_df = save_significant_neurons(neurons_df)

        # Compute mean and sem of log_pref_depth for significant neurons
        mean_log_depth = np.mean(sig_neurons_df['log_pref_depth'])
        sem = np.std(sig_neurons_df['log_pref_depth'], ddof=1) / np.sqrt(len(sig_neurons_df))

        # Save values
        mean_log_depths.append(mean_log_depth)
        sems.append(sem)

        # Get histogram
        hist_count, bin_edge = save_histogram(
            sig_neurons_df, session.name, save_path, print_figure=False
        )
        hist_counts.append(hist_count)
        bin_edges.append(bin_edge)

    # Assign all data to the dataframe
    processed['hist_counts'] = hist_counts
    processed['bin_edges'] = bin_edges
    processed['mean_log_depth'] = mean_log_depths
    processed['sem_log_depth'] = sems

    #Add mice and date names
    mice = []
    dates = []
    for i, session in processed.iterrows():
        name = session['name']
        namelist = name.split('_')
        mice.append(namelist[0])
        dates.append(namelist[1])

    processed['mouse'] = mice
    processed['date'] =  dates


    #Add exposure dates

    # Ensure 'date' is treated as a string or int for sorting (yyyymmdd format is naturally sortable)
    processed = processed.sort_values(by=['mouse', 'date']).reset_index(drop=True)

    # Create the exposure_day column by grouping by mouse and ranking the date
    processed['exposure_day'] = (
        processed
        .groupby('mouse')['date']
        .rank(method='dense')  # or method='first' if dates could repeat
        .astype(int) - 1        # Start from 0
    )

    return processed

In [ ]:

fig, ax = plt.subplots()

for mouse in tqdm(processed_mice):
    mouse_data = processed[processed['mouse'] == mouse]
    ax.errorbar(
        mouse_data['exposure_day'],
        mouse_data['mean_log_depth'],
        yerr=mouse_data['sem_log_depth'],
        label=mouse,
        capsize=3,           # small caps on error bars
        marker='o',          # dots on each point
        linestyle='-',       # connect the points
        linewidth=1
    )

ax.set_xlabel('Exposure Day')
ax.set_ylabel('Mean Log Depth')
ax.set_title('Mean Log Depth over Exposure Days by Mouse')
ax.legend()
plt.tight_layout()
plt.show()



In [ ]:
neurons_df.columns

### Plotting

In [ ]:
processed = learn.build_sessions_df(micelist)

In [ ]:
learn.plot_spearman_r_kde_by_day(processed)


Plot proportion of depth selective neurons per day

For this, 

In [ ]:
def plot_selective_proportion_over_days(processed, mice=None):
    """
    Plot proportion of depth-tuned (selective) neurons over exposure days for each mouse.

    Parameters:
    - processed: DataFrame with columns ['mouse', 'exposure_day', 'proportion_depthtuned']
    - mice: optional list of mouse IDs to plot (default: all unique mice in `processed`)
    
    Returns:
    - fig: the matplotlib figure object
    """
    if mice is None:
        mice = processed['mouse'].unique()

    fig, ax = plt.subplots()

    for mouse in tqdm(mice, desc="Plotting mice"):
        mouse_data = processed[processed['mouse'] == mouse]

        ax.plot(
            mouse_data['exposure_day'],
            mouse_data['proportion_depthtuned'],
            label=mouse,
            marker='o',
            linestyle='-',
            linewidth=1
        )

    ax.set_xlabel('Exposure Day')
    ax.set_ylabel('Proportion of Depth-Tuned Neurons')
    ax.set_title('Proportion of Selective Neurons over Exposure Days by Mouse')
    ax.legend()
    plt.tight_layout()
    plt.show()

    return fig

In [ ]:
learn.plot_selective_proportion_over_days(processed)


Plot the distributions of depth-tuned neurons

(bear in mind that gaussian KDEs are not ideal to plot non-unimodal distributions)

In [ ]:
def plot_log_pref_depth_kde_by_day(processed):
    """One subplot per exposure day showing KDEs of log_pref_depth for all mice."""
    
    exposure_days = sorted(processed['exposure_day'].unique())
    mice = sorted(processed['mouse'].unique())
    colors = plt.cm.tab10(np.linspace(0, 1, len(mice)))  # consistent colors per mouse
    mouse_color_map = dict(zip(mice, colors))

    fig, axes = plt.subplots(len(exposure_days), 1, figsize=(10, 2.5 * len(exposure_days)), sharex=True)

    if len(exposure_days) == 1:
        axes = [axes]  # Make iterable if only one subplot

    for ax, day in zip(axes, exposure_days):
        day_data = processed[processed['exposure_day'] == day]

        for _, row in day_data.iterrows():
            mouse = row['mouse']
            color = mouse_color_map[mouse]
            dist = row['log_pref_depth']

            if len(dist) < 2:
                continue  # can't KDE on 1 point

            kde = gaussian_kde(dist)
            x_range = np.linspace(min(dist), max(dist), 200)
            kde_vals = kde(x_range)

            # Plot KDE
            ax.plot(x_range, kde_vals, label=mouse, color=color, alpha=0.7)

            # Plot vertical line at median
            ax.axvline(np.median(dist), color=color, linestyle='--', alpha=0.7)

        ax.set_ylabel(f"Day {day}")
        ax.grid(True)

    axes[-1].set_xlabel("log(pref depth)")
    axes[0].set_title("KDE of log(pref depth) by Mouse for Each Exposure Day")

    # Legend: only once
    handles = [plt.Line2D([0], [0], color=mouse_color_map[m], label=m) for m in mice]
    axes[0].legend(handles=handles, title="Mouse", bbox_to_anchor=(1.05, 1), loc="upper left")

    plt.tight_layout()
    plt.show()

    return fig

In [ ]:
plot_log_pref_depth_kde_by_day(processed)

## Quantofying significance of differences

First, we look at the distribution of spearman's r's, and compare for each mouse if it changes over days 

In [ ]:
mice = None
from scipy.stats import mannwhitneyu

def test_changes_over_days(processed, property = 'spearman_r_dist', mice = None):
        if mice is None:
                mice = processed['mouse'].unique()

        p_values = []
        u_values = []

        for mouse in mice:
                mouse_data = processed[processed['mouse']==mouse]

                day_range = list(range(1, max(mouse_data['exposure_day'])+1)) #To compare curr day with the day before
                mouse_p_values = []
                mouse_u_values = []

                for day in day_range:
                        spearmans_today = np.concatenate(mouse_data[mouse_data['exposure_day'] == day]['spearman_r_dist'].values)
                        spearmans_yesterday = np.concatenate(mouse_data[mouse_data['exposure_day'] == day - 1]['spearman_r_dist'].values)

                        res = mannwhitneyu(spearmans_today, spearmans_yesterday)

                        mouse_p_values.append(res.pvalue)
                        mouse_u_values.append(res.statistic)

                p_values.append(mouse_p_values)
                u_values.append(mouse_u_values)

        return p_values, u_values

In [ ]:
def plot_pvalues_over_days(p_values, mice):
    """
    Plot p-values over exposure days for each mouse.

    Parameters:
    - p_values: list of lists, one per mouse
    - mice: list of mouse IDs in same order as p_values
    """
    fig, ax = plt.subplots(figsize=(12, 4))

    for mouse_pvals, mouse in zip(p_values, mice):
        ax.plot(
            range(1, len(mouse_pvals) + 1),
            mouse_pvals,
            label=mouse,
            marker='o',
            linestyle='-',
        )

    ax.axhline(0.05, color='red', linestyle='--', linewidth=1, label='p = 0.05')
    ax.set_xlabel('Day')
    ax.set_ylabel('Mann-Whitney U p-value')
    ax.set_title('Day-to-Day Spearman r Distribution Changes (Mann-Whitney U)')
    ax.legend(title='Mouse')
    ax.set_yscale('log')  # Optional: log scale for better visibility if p-values vary widely
    ax.grid(True)
    plt.tight_layout()
    plt.show()

    return fig

In [ ]:
plot_pvalues_over_days(p_values, mice)

# Cells across days

Found through ROICaT, going to test what is going on with it

In [ ]:
roicat_sessions = {
    'PZAG16.3c': ['S20250219', 'S20250313'], 
    'PZAG17.3a': ['S20250227', 'S20250228', 'S20250303', 'S20250305', 'S20250306'], 
    'PZAG16.3b': ['S20250224', 'S20250225', 'S20250226', 'S20250310', 'S20250313'], 
    'PZAH17.1e': ['S20250305', 'S20250311']
}


mouse = 'PZAG16.3c'
PROJECT='colasa_3d-vision_revisions'
flz_session = flz.get_flexilims_session(PROJECT)

processed_root = flz.get_data_root('processed', 
                                   project=PROJECT, 
                                   flexilims_session=flz_session
                                  )
processed = learn.find_processed_sessions(mouse)

roicat_dict = learn.load_roicat_data(mouse)

roicat_processed = learn.select_roicat_sessions(processed, mouse, roicat_sessions)
roicat_neuronsdf = learn.generate_roicat_neuronsdf(roicat_processed, roicat_dict)
sessionids = [f'{mouse}_{roicat_sessions[mouse][0]}', f'{mouse}_{roicat_sessions[mouse][1]}']
matched_df = learn.match_two_sessions(roicat_neuronsdf, sessionids)
useful_matched = learn.filter_matched_df(matched_df)


Below: a scatterplot with depth of the same neuron on two different sessions, a regression line and confidence interval on the slope and R2 of the fit

In [ ]:
figure, data = plot_pref_depth_scatter(useful_matched, show = True)

In [ ]:


def plot_pref_depth_scatter(useful_matched, use_log=True, show=False):
    """
    Scatter of preferred depth between Session 0 and Session 1 with regression lines.
    - use_log=True: uses log10(pref_depth_*). Non-positive or non-finite values are dropped.
    - use_log=False: uses raw pref_depth_*.
    Overlays regression fits for subsets filtered by depth_tuning_test_rsq_closedloop_running > 0.2, 0.4, 0.6
    Returns: fig, dict of {threshold: (slope, intercept, r2)}
    """

    x_raw = useful_matched["pref_depth_session0"].to_numpy()
    y_raw = useful_matched["pref_depth_session1"].to_numpy()

    if use_log:
        mask = np.isfinite(x_raw) & np.isfinite(y_raw) & (x_raw > 0) & (y_raw > 0)
        x = np.log10(x_raw[mask])
        y = np.log10(y_raw[mask])
        xlabel = "Log10 Preferred Depth Session 0"
        ylabel = "Log10 Preferred Depth Session 1"
        title = "Log Preferred Depths of Matched Neurons Across Two Sessions"
    else:
        mask = np.isfinite(x_raw) & np.isfinite(y_raw)
        x = x_raw[mask]
        y = y_raw[mask]
        xlabel = "Preferred Depth Session 0"
        ylabel = "Preferred Depth Session 1"
        title = "Preferred Depths of Matched Neurons Across Two Sessions"

    rsq_all = useful_matched.loc[mask, "depth_tuning_test_rsq_closedloop_running"].to_numpy()

    fig, ax = plt.subplots(figsize=(6, 5))
    sc = ax.scatter(x, y, alpha=0.5, c=rsq_all, cmap="viridis", s=20)

    # Main regression line (all points)
    m, b = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 100)
    ax.plot(xs, m * xs + b, color="red", label=f"All (R²={np.corrcoef(x,y)[0,1]**2:.2f})")

    # Regression lines for subsets
    thresholds = [0.2, 0.4, 0.6]
    colors = ["orange", "green", "blue"]
    fits = {}

    for thr, col in zip(thresholds, colors):
        submask = rsq_all > thr
        if submask.sum() < 2:
            continue  # skip if not enough points
        xx = x[submask]
        yy = y[submask]
        m_thr, b_thr = np.polyfit(xx, yy, 1)
        r_thr = np.corrcoef(xx, yy)[0, 1]
        r2_thr = r_thr**2
        ax.plot(xs, m_thr * xs + b_thr, color=col, linestyle="--",
                label=f"R²>{thr} (R²={r2_thr:.2f})")
        fits[thr] = (m_thr, b_thr, r2_thr)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()

    # Add colorbar
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label("Depth fit R²")

    fig.tight_layout()
    if show:
        plt.show()

    return fig, fits


In [ ]:
connect_figure = learn.plot_connect_depth_preference_across_sessions(roicat_neuronsdf, useful_matched, sessionids)


In [ ]:
iscell =  np.load('/nemo/lab/znamenskiyp/home/shared/projects/colasa_3d-vision_revisions/PZAH17.1e/S20250311/suite2p_rois_0/plane0/iscell.npy')

In [ ]:
iscell[:,0]

In [ ]:
list = [len(roicat_dict['labels_bySession'][key]) for key in roicat_dict['labels_bySession'].keys()]
list

In [ ]:
sessionids = [ 'PZAG17.3a_S20250303', 'PZAG17.3a_S20250305' ]

matched_df = learn.match_two_sessions(roicat_neuronsdf, roicat_dict, sessionids)

In [ ]:
idx = roicat_processed.index[roicat_processed['name'] == 'PZAG17.3a_S20250303']
idx

In [ ]:
def load_roicat_data(mouse):
    base_path = '/nemo/lab/znamenskiyp/home/shared/projects/colasa_3d-vision_revisions'
    roicat_path = os.path.join(base_path, mouse, 'ROICaT', f'{mouse}.tracking.results_clusters.json')
    
    with open(roicat_path, 'r') as f:
        roicat_dict = json.load(f)
    
    print(f"Loaded ROICaT data for {mouse}")
    print(f"Data type: {type(roicat_dict)}")
    if isinstance(roicat_dict, dict):
        print(f"Top-level keys: {list(roicat_dict.keys())}")
    
    return roicat_dict

Okay, now we load all sessions for a single mouse and add the cluster labels. 

We need to note down which session corresponds to which index in ROICaT

In [ ]:
roicat_sessions = {
    'PZAG16.3c': ['S20250219', 'S20250313'], 
    'PZAG17.3a': ['S20250227', 'S20250228', 'S20250303', 'S20250305', 'S20250306'], 
    'PZAG16.3b': ['S20250224', 'S20250225', 'S20250226', 'S20250310', 'S20250313'], 
    'PZAH17.1e': ['S20250305', 'S20250311']
}

In [ ]:
PROJECT='colasa_3d-vision_revisions'
flz_session = flz.get_flexilims_session(PROJECT)

processed_root = flz.get_data_root('processed', 
                                   project=PROJECT, 
                                   flexilims_session=flz_session
                                  )


In [ ]:
mouse = 'PZAG16.3b'

processed = learn.find_processed_sessions(mouse)

In [ ]:
processed

def select_roicat_sessions(processed, mouse, roicat_sessions):

    mouse_sessions = roicat_sessions[mouse]

    session_names = []
    for session in mouse_sessions:
        session_names.append(f'{mouse}_{session}')

    roicat_processed = processed[processed['name'].isin(session_names)]

    return roicat_processed

roicat_processed = select_roicat_sessions(processed, mouse, roicat_sessions)


def generate_roicat_neuronsdf(roicat_processed):
    roicat_neuronsdf = []
    for i, session in tqdm(roicat_processed.iterrows()):
        neurons_path = processed_root / session.path / 'neurons_df.pickle'

        with open(neurons_path, 'rb') as f:
            neurons_df = learn.NumpyFixUnpickler(f).load()
            roicat_neuronsdf.append(neurons_df)

    return roicat_neuronsdf

roicat_neuronsdf = generate_roicat_neuronsdf(roicat_processed)


In [ ]:
processed['name']

In [ ]:
roicat_dict = load_roicat_data(mouse)



In [ ]:
len(roicat_neuronsdf)


In [ ]:
print(len(roicat_neuronsdf[1]))
len(roicat_dict['labels_bySession'])

In [ ]:
roicat_dict['labels_bySession'][-4]

Weird mismatch between the ROICAT sessions before and the sessions now, why is that? Has one more session been added to the system? Which one? Why? I thought they were hard-coded by me

In [ ]:
len(roicat_dict['quality_metrics']['sample_silhouette'])

Add the cluster labels to neuronsdf

In [ ]:
def add_cluster_labels(roicat_neuronsdf, roicat_dict):

    start_neuron = 0        # running index over neurons
    start_cluster = 0       # running index over clusters

    for s, ndf in enumerate(roicat_neuronsdf):
        # ----------------- 1.  add cluster_id (already OK) -----------------
        ndf['cluster_id'] = roicat_dict['labels_bySession'][s]

        # ----------------- 2.  slice the two global arrays -----------------
        # how many neurons and how many clusters in this session?
        n_neurons  = len(ndf)
        n_clusters = len(np.unique(ndf['cluster_id']))

        # neuron-level quality metric (one per neuron)
        end_neuron = start_neuron + n_neurons
        ndf['sample_silhouette'] = roicat_dict['quality_metrics']['sample_silhouette'][start_neuron:end_neuron]

        # cluster-level quality metric (one per cluster)
        end_cluster = start_cluster + n_clusters
        clust_sil   = roicat_dict['quality_metrics']['cluster_silhouette'][start_cluster:end_cluster]

        # ----------------- 3.  broadcast cluster silhouette to every neuron -----------------
        # map cluster_id → silhouette
        clust_sil_map = dict(enumerate(clust_sil))          # assumes clusters are indexed 0..n_clusters-1
        ndf['cluster_silhouette'] = ndf['cluster_id'].map(clust_sil_map)

        # ----------------- 4.  advance the running indices -----------------
        start_neuron  = end_neuron
        start_cluster = end_cluster
    
    # Grab the two sessions
    df0 = roicat_neuronsdf[0]          # session 0
    df1 = roicat_neuronsdf[1]          # session 1

    # 1) clusters present in *both* sessions
    common_clusters = np.intersect1d(df0['cluster_id'].unique(),
                                    df1['cluster_id'].unique())

    rows = []
    for cid in common_clusters:
        # row for this cluster in each session
        row0 = df0[df0['cluster_id'] == cid]
        row1 = df1[df1['cluster_id'] == cid]
        
        # guard: skip clusters that have more than one neuron per session
        if len(row0) != 1 or len(row1) != 1:
            continue                  # or handle as you wish
        
        row0 = row0.iloc[0]
        row1 = row1.iloc[0]
        
        rows.append({
            'cluster_id'                : cid,
            
            'neuron_idx_session0'       : row0.name,   # DataFrame index
            'neuron_idx_session1'       : row1.name,
            
            'sample_sil_session0'       : row0['sample_silhouette'],
            'sample_sil_session1'       : row1['sample_silhouette'],
            
            'cluster_silhouette'        : row0['cluster_silhouette'],  # same for both
            
            'spearman_r_session0'       : row0['depth_tuning_test_spearmanr_rval_closedloop'],
            'spearman_r_session1'       : row1['depth_tuning_test_spearmanr_rval_closedloop'],
            
            'pref_depth_session0'       : row0['preferred_depth_closedloop'],
            'pref_depth_session1'       : row1['preferred_depth_closedloop'],
        })

    matched_df = pd.DataFrame(rows).reset_index(drop=True)
    matched_df.head()

    return matched_df


In [ ]:
start_neuron = 0        # running index over neurons
start_cluster = 0       # running index over clusters

for s, ndf in enumerate(roicat_neuronsdf):
    # ----------------- 1.  add cluster_id (already OK) -----------------
    ndf['cluster_id'] = roicat_dict['labels_bySession'][s]

    # ----------------- 2.  slice the two global arrays -----------------
    # how many neurons and how many clusters in this session?
    n_neurons  = len(ndf)
    n_clusters = len(np.unique(ndf['cluster_id']))

    # neuron-level quality metric (one per neuron)
    end_neuron = start_neuron + n_neurons
    ndf['sample_silhouette'] = roicat_dict['quality_metrics']['sample_silhouette'][start_neuron:end_neuron]

    # cluster-level quality metric (one per cluster)
    end_cluster = start_cluster + n_clusters
    clust_sil   = roicat_dict['quality_metrics']['cluster_silhouette'][start_cluster:end_cluster]

    # ----------------- 3.  broadcast cluster silhouette to every neuron -----------------
    # map cluster_id → silhouette
    clust_sil_map = dict(enumerate(clust_sil))          # assumes clusters are indexed 0..n_clusters-1
    ndf['cluster_silhouette'] = ndf['cluster_id'].map(clust_sil_map)

    # ----------------- 4.  advance the running indices -----------------
    start_neuron  = end_neuron
    start_cluster = end_cluster

    

In [ ]:

# Grab the two sessions
df0 = roicat_neuronsdf[0]          # session 0
df1 = roicat_neuronsdf[1]          # session 1

# 1) clusters present in *both* sessions
common_clusters = np.intersect1d(df0['cluster_id'].unique(),
                                 df1['cluster_id'].unique())

rows = []
for cid in common_clusters:
    # row for this cluster in each session
    row0 = df0[df0['cluster_id'] == cid]
    row1 = df1[df1['cluster_id'] == cid]
    
    # guard: skip clusters that have more than one neuron per session
    if len(row0) != 1 or len(row1) != 1:
        continue                  # or handle as you wish
    
    row0 = row0.iloc[0]
    row1 = row1.iloc[0]
    
    rows.append({
        'cluster_id'                : cid,
        
        'neuron_idx_session0'       : row0.name,   # DataFrame index
        'neuron_idx_session1'       : row1.name,
        
        'sample_sil_session0'       : row0['sample_silhouette'],
        'sample_sil_session1'       : row1['sample_silhouette'],
        
        'cluster_silhouette'        : row0['cluster_silhouette'],  # same for both
        
        'spearman_r_session0'       : row0['depth_tuning_test_spearmanr_rval_closedloop'],
        'spearman_r_session1'       : row1['depth_tuning_test_spearmanr_rval_closedloop'],
        
        'pref_depth_session0'       : row0['preferred_depth_closedloop'],
        'pref_depth_session1'       : row1['preferred_depth_closedloop'],
    })

matched_df = pd.DataFrame(rows).reset_index(drop=True)
matched_df.head()


In [ ]:
useful_matched = matched_df[
    (matched_df['cluster_silhouette'] > 0.2) &
    (matched_df['sample_sil_session0'] > 0.1) &
    (matched_df['sample_sil_session1'] > 0.1)
]
# ----  INPUTS  ---------------------------------------------------------
df0 = roicat_neuronsdf[0]      # session-0 neuron dataframe
df1 = roicat_neuronsdf[1]      # session-1 neuron dataframe
# matched_df is the table we built earlier
# ----------------------------------------------------------------------

# Helper: get log depth for an arbitrary dataframe
def get_log_depth(df, raw_key='preferred_depth_closedloop', log_key='log_pref_depth'):
    if log_key in df.columns:
        return df[log_key].values
    if raw_key in df.columns:
        return np.log10(df[raw_key].values)              # natural-log; use np.log10 if you prefer
    raise KeyError("Depth column not found")

y0 = get_log_depth(df0)
y1 = get_log_depth(df1)

# add a tiny horizontal jitter so points don’t overlap perfectly
x0 = np.random.normal(loc=0, scale=0.04, size=len(y0))
x1 = np.random.normal(loc=1, scale=0.04, size=len(y1))

fig, ax = plt.subplots(figsize=(4, 6))

ax.scatter(x0, y0, alpha=0.4, color='steelblue', s=10)
ax.scatter(x1, y1, alpha=0.4, color='coral',    s=10)

# Plot connecting lines for every matched cluster
for _, row in useful_matched.iterrows():
    # guarantee log depth
    d0 = np.log10(row['pref_depth_session0'])
    d1 = np.log10(row['pref_depth_session1'])
    ax.plot([0, 1], [d0, d1], color='gray', alpha=0.6, linewidth=1)

ax.set_xticks([0, 1])
ax.set_xticklabels(['Session 0', 'Session 1'])
ax.set_ylabel('log (pref-depth)')
ax.set_title('Depth preference of neurons across sessions\n(connected lines = same cluster)')

# make it pretty
ax.spines[['right', 'top']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
def plot_depth_preference_across_sessions(roicat_neuronsdf, 
                                          matched_df, 
                                          cluster_silhouette_threshold=0.2, 
                                          sample_silhouette_threshold=0.1, 
                                          filter_extreme_depths = False):
    """
    Plot depth preference of neurons across two sessions, with connecting lines for 
    matched clusters.
    
    Parameters:
    - roicat_neuronsdf: list of DataFrames for each session
    - matched_df: DataFrame containing matched clusters between sessions
    """
    useful_matched = matched_df[
    (matched_df['cluster_silhouette'] > cluster_silhouette_threshold) &
    (matched_df['sample_sil_session0'] > sample_silhouette_threshold) &
    (matched_df['sample_sil_session1'] > sample_silhouette_threshold)
    ]

    if filter_extreme_depths:
        useful_matched = useful_matched[
        (np.log10(useful_matched['pref_depth_session0']) < 1) &
        (np.log10(useful_matched['pref_depth_session0']) > -1.5) &
        (np.log10(useful_matched['pref_depth_session1']) < 1) &
        (np.log10(useful_matched['pref_depth_session1']) > -1.5)
        ]

    df0 = roicat_neuronsdf[0]
    df1 = roicat_neuronsdf[1]

    y0 = get_log_depth(df0)
    y1 = get_log_depth(df1)

    # add a tiny horizontal jitter so points don’t overlap perfectly
    x0 = np.random.normal(loc=0, scale=0.04, size=len(y0))
    x1 = np.random.normal(loc=1, scale=0.04, size=len(y1))

    fig, ax = plt.subplots(figsize=(4, 6))

    ax.scatter(x0, y0, alpha=0.4, color='steelblue', s=10)
    ax.scatter(x1, y1, alpha=0.4, color='coral',    s=10)

    # Plot connecting lines for every matched cluster
    for _, row in useful_matched.iterrows():
        # guarantee log depth
        d0 = np.log10(row['pref_depth_session0'])
        d1 = np.log10(row['pref_depth_session1'])
        ax.plot([0, 1], [d0, d1], color='gray', alpha=0.6, linewidth=1)

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Session 0', 'Session 1'])
    ax.set_ylabel('log (pref-depth)')
    if filter_extreme_depths:
        ax.set_title('Depth preference of neurons across sessions\n(connected lines = same cluster)\nExtreme depths filtered out')
    else:
        ax.set_title('Depth preference of neurons across sessions\n(connected lines = same cluster)')

    # make it pretty
    ax.spines[['right', 'top']].set_visible(False)
    plt.tight_layout()
    plt.show()    

    return fig        

In [ ]:
useful_matched = matched_df[
    (matched_df['cluster_silhouette'] > 0.2) &
    (matched_df['sample_sil_session0'] > 0.1) &
    (matched_df['sample_sil_session1'] > 0.1)
]

useful_matched_no_extremes = useful_matched[
    (np.log10(useful_matched['pref_depth_session0']) < 1) &
    (np.log10(useful_matched['pref_depth_session0']) > -1.5) &
    (np.log10(useful_matched['pref_depth_session1']) < 1) &
    (np.log10(useful_matched['pref_depth_session1']) > -1.5)
]

# ----  INPUTS  ---------------------------------------------------------
df0 = roicat_neuronsdf[0]      # session-0 neuron dataframe
df1 = roicat_neuronsdf[1]      # session-1 neuron dataframe
# matched_df is the table we built earlier
# ----------------------------------------------------------------------

# Helper: get log depth for an arbitrary dataframe
def get_log_depth(df, raw_key='preferred_depth_closedloop', log_key='log_pref_depth'):
    if log_key in df.columns:
        return df[log_key].values
    if raw_key in df.columns:
        return np.log10(df[raw_key].values)              # natural-log; use np.log10 if you prefer
    raise KeyError("Depth column not found")

y0 = get_log_depth(df0)
y1 = get_log_depth(df1)

# add a tiny horizontal jitter so points don’t overlap perfectly
x0 = np.random.normal(loc=0, scale=0.04, size=len(y0))
x1 = np.random.normal(loc=1, scale=0.04, size=len(y1))

fig, ax = plt.subplots(figsize=(4, 6))

ax.scatter(x0, y0, alpha=0.4, color='steelblue', s=10)
ax.scatter(x1, y1, alpha=0.4, color='coral',    s=10)

# Plot connecting lines for every matched cluster
for _, row in useful_matched_no_extremes.iterrows():
    # guarantee log depth
    d0 = np.log10(row['pref_depth_session0'])
    d1 = np.log10(row['pref_depth_session1'])
    ax.plot([0, 1], [d0, d1], color='gray', alpha=0.6, linewidth=1)

ax.set_xticks([0, 1])
ax.set_xticklabels(['Session 0', 'Session 1'])
ax.set_ylabel('log (pref-depth)')
ax.set_title('Depth preference of neurons across sessions\n(connected lines = same cluster)\nExtreme depths filtered out')

# make it pretty
ax.spines[['right', 'top']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
useful_matched['difference'] = useful_matched['pref_depth_session1']-useful_matched['pref_depth_session0']

In [ ]:
# Initialize
n_shuffles = 20
null_differences = np.zeros((len(useful_matched), n_shuffles))

# Extract original depth arrays
pref0 = useful_matched['pref_depth_session0'].values
pref1 = useful_matched['pref_depth_session1'].values

# Shuffle independently
for i in range(n_shuffles):
    shuffled_pref0 = np.random.permutation(pref0)
    shuffled_pref1 = np.random.permutation(pref1)
    
    null_differences[:, i] = shuffled_pref1 - shuffled_pref0

In [ ]:


# diffs: real differences (Session 1 - Session 0)
diffs = useful_matched['difference']

plt.figure(figsize=(6,4))

# 1. Plot histogram of the real absolute differences and get bins
counts, bins, patches = plt.hist(np.abs(diffs), bins=50, color='blue', alpha=0.7)
plt.xscale('log')

# 2. Overlay very light red lines for each null distribution
for i in range(null_differences.shape[1]):  # 20 shuffled versions
    shuffled_diff = null_differences[:, i]
    plt.hist(np.abs(shuffled_diff), bins=bins, color='red', alpha=0.5, histtype='step')

# 3. No need to manually control xlim — bins already control it nicely!
# plt.xlim(...)  # optional if you still want tighter control

plt.title('Absolute Difference in depth preference (real vs null)')
plt.xlabel('Absolute Difference in Depth (m)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()



In [ ]:
def plot_difference_histogram(useful_matched, n_shuffles=50):
    """
    Plot histogram of absolute differences in depth preference between sessions,
    overlaying histograms of null distributions from shuffled data.

    Parameters:     
    - useful_matched: DataFrame containing matched clusters with depth preferences, 
                      filtered by silhouette scores
    - n_shuffles: Number of shuffles to create null distributions (default: 50)

    Returns:
    - fig: matplotlib Figure object (so the caller can save it)
    """

    # Initialize
    null_differences = np.zeros((len(useful_matched), n_shuffles))

    # Extract original depth arrays
    pref0 = useful_matched['pref_depth_session0'].values
    pref1 = useful_matched['pref_depth_session1'].values

    # Shuffle independently
    for i in range(n_shuffles):
        shuffled_pref0 = np.random.permutation(pref0)
        shuffled_pref1 = np.random.permutation(pref1)
        null_differences[:, i] = shuffled_pref1 - shuffled_pref0

    # diffs: real differences (Session 1 - Session 0)
    diffs = useful_matched['difference']

    # Create figure/axes explicitly
    fig, ax = plt.subplots(figsize=(6,4))

    # 1. Plot histogram of the real absolute differences and get bins
    counts, bins, patches = ax.hist(np.abs(diffs), bins=50, color='blue', alpha=0.7)
    ax.set_xscale('log')

    # 2. Overlay very light red lines for each null distribution
    for i in range(null_differences.shape[1]):
        shuffled_diff = null_differences[:, i]
        ax.hist(np.abs(shuffled_diff), bins=bins, color='red', alpha=0.5, histtype='step')

    ax.set_title('Absolute Difference in depth preference (real vs null)')
    ax.set_xlabel('Absolute Difference in Depth (m)')
    ax.set_ylabel('Frequency')

    fig.tight_layout()

    return fig

In [ ]:
# Grab the two sessions
df0 = roicat_neuronsdf[0]          # session 0
df3 = roicat_neuronsdf[3]          # session 3   <<<----- now session 3

# 1) clusters present in *both* sessions
common_clusters = np.intersect1d(df0['cluster_id'].unique(),
                                 df3['cluster_id'].unique())

rows = []
for cid in common_clusters:
    # row for this cluster in each session
    row0 = df0[df0['cluster_id'] == cid]
    row3 = df3[df3['cluster_id'] == cid]
    
    # guard: skip clusters that have more than one neuron per session
    if len(row0) != 1 or len(row3) != 1:
        continue
    
    row0 = row0.iloc[0]
    row3 = row3.iloc[0]
    
    rows.append({
        'cluster_id'                : cid,
        
        'neuron_idx_session0'       : row0.name,
        'neuron_idx_session3'       : row3.name,
        
        'sample_sil_session0'       : row0['sample_silhouette'],
        'sample_sil_session3'       : row3['sample_silhouette'],
        
        'cluster_silhouette'        : row0['cluster_silhouette'],
        
        'spearman_r_session0'       : row0['depth_tuning_test_spearmanr_rval_closedloop'],
        'spearman_r_session3'       : row3['depth_tuning_test_spearmanr_rval_closedloop'],
        
        'pref_depth_session0'       : row0['preferred_depth_closedloop'],
        'pref_depth_session3'       : row3['preferred_depth_closedloop'],
    })

matched_df = pd.DataFrame(rows).reset_index(drop=True)

# Filter useful matches
useful_matched = matched_df[
    (matched_df['cluster_silhouette'] > 0.2) &
    (matched_df['sample_sil_session0'] > 0.1) &
    (matched_df['sample_sil_session3'] > 0.1)
]

# ---- Plotting ----
def get_log_depth(df, raw_key='preferred_depth_closedloop', log_key='log_pref_depth'):
    if log_key in df.columns:
        return df[log_key].values
    if raw_key in df.columns:
        return np.log10(df[raw_key].values)
    raise KeyError("Depth column not found")

y0 = get_log_depth(df0)
y3 = get_log_depth(df3)

x0 = np.random.normal(loc=0, scale=0.04, size=len(y0))
x3 = np.random.normal(loc=1, scale=0.04, size=len(y3))

fig, ax = plt.subplots(figsize=(4, 6))

ax.scatter(x0, y0, alpha=0.4, color='steelblue', s=10)
ax.scatter(x3, y3, alpha=0.4, color='coral', s=10)

# Plot connecting lines for each matched neuron
for _, row in useful_matched.iterrows():
    d0 = np.log10(row['pref_depth_session0'])
    d3 = np.log10(row['pref_depth_session3'])
    ax.plot([0, 1], [d0, d3], color='gray', alpha=0.6, linewidth=1)

ax.set_xticks([0, 1])
ax.set_xticklabels(['Session 0', 'Session 3'])  # <<< updated x-tick labels
ax.set_ylabel('log (pref-depth)')
ax.set_title('Depth preference of neurons across sessions\n(connected lines = same cluster)')

ax.spines[['right', 'top']].set_visible(False)
plt.tight_layout()
plt.show()



In [ ]:
useful_matched['difference'] = useful_matched['pref_depth_session3']-useful_matched['pref_depth_session0']

# Initialize
n_shuffles = 20
null_differences = np.zeros((len(useful_matched), n_shuffles))

# Extract original depth arrays
pref0 = useful_matched['pref_depth_session0'].values
pref3 = useful_matched['pref_depth_session3'].values

# Shuffle independently
for i in range(n_shuffles):
    shuffled_pref0 = np.random.permutation(pref0)
    shuffled_pref3 = np.random.permutation(pref3)
    
    null_differences[:, i] = shuffled_pref3 - shuffled_pref0

# diffs: real differences (Session 1 - Session 0)
diffs = useful_matched['difference']

plt.figure(figsize=(6,4))

# 1. Plot histogram of the real absolute differences and get bins
counts, bins, patches = plt.hist(np.abs(diffs), bins=50, color='blue', alpha=0.7)
plt.xscale('log')

# 2. Overlay very light red lines for each null distribution
for i in range(null_differences.shape[1]):  # 20 shuffled versions
    shuffled_diff = null_differences[:, i]
    plt.hist(np.abs(shuffled_diff), bins=bins, color='red', alpha=0.5, histtype='step')

# 3. No need to manually control xlim — bins already control it nicely!
# plt.xlim(...)  # optional if you still want tighter control

plt.title('Difference in depth preference 0-3 (real vs null)')
plt.xlabel('Difference in Depth (m)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

# Macro comparisons

We are running a hierarchical bootstrap approach. This means that we are sampling hierarchically for each comparison, first mice and then cells. 

In [ ]:
micelist = ['PZAG16.3b', 'PZAG16.3c', 'PZAH17.1e']

In [ ]:
sessions_df = learn.build_sessions_df(micelist)

In [ ]:
sessions_df

CAREFUL! Now, the preferred depth is stored for significant neurons, while the spearman's r is stored for all neurons. 

In [ ]:

def hierarchical_bootstrap(quantity, day0, day1, sessions_df, statistic = 'median'):

    # Find how many mice are on each day

    df_day0 = sessions_df[sessions_df['exposure_day']==day0]
    mice_day0 = set(df_day0['mouse'])

    df_day1 = sessions_df[sessions_df['exposure_day']==day1]
    mice_day1 = set(df_day1['mouse'])

    # Find out how many cells are there per mouse on each day

    cells_day0 = {}
    for m in mice:
        series = df_day0.loc[df_day0['mouse'] == m, quantity]
        # series may be scalar per row OR array per row
        if np.isscalar(series.iloc[0]):                 # ordinary numeric column
            cells_day0[m] = series.values
        else:                                           # column of arrays/lists
            cells_day0[m] = np.concatenate(series.values)

    cells_day1 = {}
    for m in mice:
        series = df_day1.loc[df_day1['mouse'] == m, quantity]
        # series may be scalar per row OR array per row
        if np.isscalar(series.iloc[0]):                 # ordinary numeric column
            cells_day1[m] = series.values
        else:                                           # column of arrays/lists
            cells_day1[m] = np.concatenate(series.values)

    #start the number of repetitions

    nrepeats = 1000
    boot_diffs = np.zeros(nrepeats)


    for i in tqdm(np.arange(nrepeats)):

        #sample the mice
        mice_boot_0 = random.choices(list(mice_day0), k=len(mice_day0))
        mice_boot_1 = random.choices(list(mice_day1), k=len(mice_day1))

        #sample the cells
        cells_boot_0 = {}
        for mouse in mice_boot_0:
            cells_boot_0[mouse] = random.choices(list(cells_day0[mouse]), k=len(cells_day0[mouse]))

        cells_boot_1 = {}
        for mouse in mice_boot_1:
            cells_boot_1[mouse] = random.choices(list(cells_day1[mouse]), k=len(cells_day1[mouse]))

        
        # concatenate all sampled cells across mice
        all_cells_boot_0 = [cell for cells in cells_boot_0.values() for cell in cells]
        all_cells_boot_1 = [cell for cells in cells_boot_1.values() for cell in cells]

        # compute the statistic
        if statistic =='median':
            median0 = np.median(all_cells_boot_0)
            median1 = np.median(all_cells_boot_1)
        if statistic=='mean':
            median0 = np.mean(all_cells_boot_0)
            median1 = np.mean(all_cells_boot_1)
        boot_diffs[i] = median1 - median0

    # compute a 95% bootstrap CI
    ci_lower, ci_upper = np.percentile(boot_diffs, [2.5, 97.5])
    print(f"95% CI for Δmean: [{ci_lower:.3f}, {ci_upper:.3f}]")

    # compute a two‐sided p‐value
    p_val = 2 * min((boot_diffs <= 0).mean(), (boot_diffs >= 0).mean())
    print(f"Approx. p‐value: {p_val:.3f}")

    return {
        "boot_diffs": boot_diffs,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "p_val": p_val
    }

In [ ]:
other_days = [1, 2, 3, 4]

tests = []

for day in other_days:
    tests.append(hierarchical_bootstrap('spearman_r_dist', 0, day, sessions_df))

In [ ]:

def plot_comparison(tests, quantity = 'spearman r'):

    # Example: Assume 'tests' is already defined
    fig, ax = plt.subplots(figsize=(10, 6))

    # Extract boot_diffs for all tests
    boot_diffs_all = [test['boot_diffs'] for test in tests]

    # Plot boxplots
    box = ax.boxplot(boot_diffs_all, patch_artist=True, showfliers=False, whis=(5, 95))

    # Color each box with a different color
    colors = ['royalblue', 'seagreen', 'darkorange', 'crimson']
    for patch, color in zip(box['boxes'], colors):
        patch.set_facecolor(color)

    # Draw a horizontal dashed line at 0
    ax.axhline(0, color='black', linestyle='--', linewidth=1)

    # Label x-axis
    ax.set_xticklabels([f'Day {i+1}' for i in range(len(tests))])
    ax.set_ylabel('Hierarchical bootstraped X-X0')
    ax.set_title(f'Differences in mean {quantity} with day 0')

    # Overlay scatter points for each bootstrapped dataset
    for i, boot_diffs in enumerate(boot_diffs_all):
        # Add random horizontal jitter to spread out points a bit
        x_jitter = np.random.normal(loc=0, scale=0.05, size=len(boot_diffs))
        ax.scatter(np.full_like(boot_diffs, i + 1) + x_jitter, boot_diffs,
                color='black', alpha=0.2, s=10)

    # Write p-values inside the boxes (white, centered)
    for i, (test, patch) in enumerate(zip(tests, box['boxes'])):
        p_val = test['p_val']
        # Get box height from vertices
        path = patch.get_path()
        verts = path.vertices
        y_bottom = verts[1,1]
        y_top = verts[2,1]
        y_center = (y_bottom + y_top) / 2
        ax.text(i + 1, y_center, f'p = {p_val:.3f}', color='white',
                ha='center', va='center', fontsize=10, weight='bold')

    plt.tight_layout()
    plt.show()

    return fig, ax

fig, ax = plot_comparison(tests)



We do the same with the particular depth tuning

In [ ]:
other_days = [1, 2, 3, 4]

tests = []

for day in other_days:
    tests.append(hierarchical_bootstrap('log_pref_depth', 0, day, sessions_df))

In [ ]:
fig, ax = plot_comparison(tests, 'mean_log_depth')

And remake the KDEs

In [ ]:
relevant_days = sessions_df[sessions_df['exposure_day'].isin([0, 1, 2, 3, 4])]

learn.plot_log_depth_over_days(relevant_days, mice = micelist)

In [ ]:
learn.plot_spearman_r_kde_by_day(relevant_days)

In [ ]:
learn.plot_selective_proportion_over_days(relevant_days, mice=micelist)

In [ ]:


def bootstrap_ci(quantity, day, sessions_df, nrepeats=1000):
    """
    Computes a bootstrap distribution and confidence interval for the mean of a variable on a given day.
    
    Args:
        quantity (str): Column name in sessions_df to bootstrap.
        day (int): The day (e.g., 0 or 1) to subset the data.
        sessions_df (pd.DataFrame): DataFrame with session data.
        nrepeats (int): Number of bootstrap iterations.

    Returns:
        dict: {
            'boot_means': array of bootstrapped means,
            'ci_lower': 2.5% percentile,
            'ci_upper': 97.5% percentile
        }
    """

    # Subset sessions to the selected day
    df_day = sessions_df[sessions_df['exposure_day'] == day]
    mice = set(df_day['mouse'])

    cells_by_mouse = {}
    for m in mice:
        series = df_day.loc[df_day['mouse'] == m, quantity]
        # series may be scalar per row OR array per row
        if np.isscalar(series.iloc[0]):                 # ordinary numeric column
            cells_by_mouse[m] = series.values
        else:                                           # column of arrays/lists
            cells_by_mouse[m] = np.concatenate(series.values)


    boot_means = np.zeros(nrepeats)

    for i in tqdm(range(nrepeats)):
        # Resample mice with replacement
        sampled_mice = random.choices(list(mice), k=len(mice))

        # Resample cells within mice and flatten
        sampled_cells = []
        for mouse in sampled_mice:
            mouse_cells = cells_by_mouse[mouse]
            resampled_cells = random.choices(mouse_cells, k=len(mouse_cells))
            sampled_cells.extend(resampled_cells)

        # Compute the mean of the bootstrap sample
        boot_means[i] = np.mean(sampled_cells)

    # Compute 95% confidence interval
    ci_lower, ci_upper = np.percentile(boot_means, [2.5, 97.5])

    return {
        'boot_means': boot_means,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper
    }


In [ ]:
shares = []
days = [0, 1, 2, 3, 4]

for day in days:
    shares.append(bootstrap_ci('is_depth_neuron_dist', day, sessions_df))

In [ ]:
other_days = [1, 2, 3, 4]

tests = []

for day in other_days:
    tests.append(hierarchical_bootstrap('is_depth_neuron_dist', 0, day, sessions_df, statistic='mean'))

In [ ]:
fig, ax = plot_comparison(tests, 'is_depth_neuron_dist')

In [ ]:
def plot_interval(shares, tests, quantity = 'share of depth selective'):

    # Example: Assume 'tests' is already defined
    fig, ax = plt.subplots(figsize=(10, 6))

    # Extract boot_diffs for all tests
    boot_means_all = [share['boot_means'] for share in shares]

    # Plot boxplots
    box = ax.boxplot(boot_means_all, patch_artist=True, showfliers=False, whis=[5, 95])

    # Color each box with a different color
    colors = ['royalblue', 'seagreen', 'darkorange', 'crimson']
    for patch, color in zip(box['boxes'], colors):
        patch.set_facecolor(color)

    # Label x-axis
    ax.set_xticklabels([f'Day {i}' for i in range(len(shares))])
    ax.set_ylabel('Bootstrapped mean')
    ax.set_title(f'CI {quantity}')

    # Overlay scatter points for each bootstrapped dataset
    for i, boot_diffs in enumerate(boot_means_all):
        # Add random horizontal jitter to spread out points a bit
        x_jitter = np.random.normal(loc=0, scale=0.05, size=len(boot_diffs))
        ax.scatter(np.full_like(boot_diffs, i + 1) + x_jitter, boot_diffs,
                color='black', alpha=0.2, s=10)

    # Write p-values inside the boxes (white, centered)
    for i, (test, patch) in enumerate(zip(tests, box['boxes'][1:])):
        p_val = test['p_val']
        # Get box height from vertices
        path = patch.get_path()
        verts = path.vertices
        y_bottom = verts[1,1]
        y_top = verts[2,1]
        y_center = (y_bottom + y_top) / 2
        ax.text(i + 2, y_center, f'p = {p_val:.3f}', color='white',
                ha='center', va='center', fontsize=10, weight='bold')

    plt.tight_layout()
    plt.show()

    return fig, ax

fig, ax = plot_interval(shares, tests)

In [ ]:
shares = []
days = [0, 1, 2, 3, 4]

for day in days:
    shares.append(bootstrap_ci('log_pref_depth', day, sessions_df))


other_days = [1, 2, 3, 4]

tests = []

for day in other_days:
    tests.append(hierarchical_bootstrap('log_pref_depth', 0, day, sessions_df, statistic='mean'))

In [ ]:
fig, ax = plot_interval(shares, tests, quantity='log_pref_depth')

In [ ]:
sessions_df

In [ ]:
quantity = 'spearman_r_dist'

# Subset sessions to the selected day
df_day = sessions_df[sessions_df['exposure_day'] == day]
mice = set(df_day['mouse'])

mode = 'chatgpt'

if mode == 'chatgpt':

    cells_by_mouse = {}
    for m in mice:
        series = df_day.loc[df_day['mouse'] == m, quantity]
        # series may be scalar per row OR array per row
        if np.isscalar(series.iloc[0]):                 # ordinary numeric column
            cells_by_mouse[m] = series.values
        else:                                           # column of arrays/lists
            cells_by_mouse[m] = np.concatenate(series.values)

if mode == 'me':

    cells_by_mouse = {}
    for mouse in mice:
        mouse_df = df_day[df_day['mouse']==mouse]
        cells_by_mouse[mouse]=list(mouse_df[quantity])



In [ ]:
cells_by_mouse['PZAG16.3b']